In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
import ta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
# Definimos Todas Variáveis
tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F',    
    'Juros_BR': 'LFTS11.SA',
    'Inflacao_BR': 'IMAB11.SA'
}

In [3]:
print("📥 Baixando cotações...")
df_precos = yf.download(list(tickers.values()), period='3y')['Close']
df_precos.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
df_precos.ffill(inplace=True)
df_precos.dropna(inplace=True)

📥 Baixando cotações...


[*********************100%***********************]  9 of 9 completed


In [4]:
print("🧬 Calculando Indicadores Técnicos...")
df_features = pd.DataFrame(index=df_precos.index)

for col in df_precos.columns:
    df_features[col] = df_precos[col].pct_change()

df_features['RSI'] = ta.momentum.RSIIndicator(df_precos['Bovespa'], window=14).rsi()
sma_15 = ta.trend.SMAIndicator(df_precos['Bovespa'], window=15).sma_indicator()
df_features['Distancia_SMA15'] = (df_precos['Bovespa'] / sma_15) - 1
df_features['Bollinger_Width'] = ta.volatility.BollingerBands(df_precos['Bovespa'], window=20).bollinger_wband()

df_features.dropna(inplace=True)

time_steps = 7
colunas_features = df_features.columns

🧬 Calculando Indicadores Técnicos...


In [5]:
def classificar_retorno_3_classes(retorno):
    if retorno < -0.002: return 0       # Baixa (menor que -0.2%)
    elif -0.002 <= retorno <= 0.002: return 1 # Neutro (-0.2% a +0.2%)
    else: return 2                      # Alta (maior que +0.2%)

nomes_classes = {
    0: "📉 Baixa (menor que -0.2%)",
    1: "➖ Neutro (-0.2% a +0.2%)",
    2: "📈 Alta (maior que +0.2%)"
}

In [6]:
def createDatasetClassificacao(dataset, time_steps):
    X, y = [], []
    idx_bovespa = list(dataset.columns).index('Bovespa')
    data_array = dataset.values
    
    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        retorno_futuro = data_array[i + time_steps, idx_bovespa]
        y.append(classificar_retorno_3_classes(retorno_futuro))
        
    return np.array(X), np.array(y)

X, y = createDatasetClassificacao(df_features, time_steps)

In [7]:
# Atenção aqui: num_classes mudou para 3!
y_encoded = to_categorical(y, num_classes=3) 

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_encoded[:split], y_encoded[split:]

In [8]:
print("📏 Normalizando as Variáveis...")
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

📏 Normalizando as Variáveis...


In [9]:
time_steps = 7
colunas_features = df_features.columns

In [10]:
print("🧠 Treinando Rede Neural...")
modelo_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(time_steps, len(colunas_features))),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(3, activation='softmax') # Atenção aqui: 3 neurónios de saída!
])

C:\Users\flavi\anaconda3\envs\SeriesTemporais\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


🧠 Treinando Rede Neural...


In [11]:
modelo_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
modelo_gru.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_split=0.1, verbose=0)

if not os.path.exists('modelos'): os.makedirs('modelos')
modelo_gru.save_weights("modelos/modelo_gru_classificador_3c.weights.h5", overwrite=True)
joblib.dump(scaler_X, 'modelos/scaler_X_classificador_3c.pkl')

['modelos/scaler_X_classificador_3c.pkl']

In [12]:
print("📏 Normalizando as Variáveis...")
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

📏 Normalizando as Variáveis...


In [13]:
loss, acuracia = modelo_gru.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n--- 📊 RESULTADOS DO TREINO (3 CLASSES COM TA) ---")
print(f"Acurácia Global do Modelo: {acuracia * 100:.2f}%")


--- 📊 RESULTADOS DO TREINO (3 CLASSES COM TA) ---
Acurácia Global do Modelo: 44.74%


In [14]:
ultimos_dias = df_features.tail(time_steps).values
X_futuro_scaled = scaler_X.transform(ultimos_dias.reshape(-1, ultimos_dias.shape[-1])).reshape(1, time_steps, len(colunas_features))

# Probabilidades da Rede Neural
probabilidades = modelo_gru.predict(X_futuro_scaled, verbose=0)[0]
classe_vencedora = np.argmax(probabilidades)

# Matemática para transformar as classes em Pontos Reais
fechamento_hoje = df_precos['Bovespa'].iloc[-1]
limite_inferior = fechamento_hoje * (1 - 0.002) # -0.2%
limite_superior = fechamento_hoje * (1 + 0.002) # +0.2%

print("\n🚀 PREVISÃO PARA O PRÓXIMO DIA ÚTIL 🚀")
print(f"Cotação Base (Fechamento Hoje): {fechamento_hoje:.2f} pts")
print(f"Cenário mais provável: {nomes_classes[classe_vencedora]}")

print("\n📊 Raio-X de Probabilidades:")
for i in range(3):
    print(f"{nomes_classes[i]}: {probabilidades[i]*100:.2f}%")

print("\n🎯 O QUE ISSO SIGNIFICA EM PONTOS PARA AMANHÃ:")
print(f"🟢 Para ser ALTA: Precisa fechar ACIMA de {limite_superior:.0f} pts")
print(f"🔴 Para ser BAIXA: Precisa fechar ABAIXO de {limite_inferior:.0f} pts")
print(f"🟡 Zona NEUTRA: Ficar preso entre {limite_inferior:.0f} e {limite_superior:.0f} pts")


🚀 PREVISÃO PARA O PRÓXIMO DIA ÚTIL 🚀
Cotação Base (Fechamento Hoje): 175135.41 pts
Cenário mais provável: 📈 Alta (maior que +0.2%)

📊 Raio-X de Probabilidades:
📉 Baixa (menor que -0.2%): 24.08%
➖ Neutro (-0.2% a +0.2%): 25.97%
📈 Alta (maior que +0.2%): 49.96%

🎯 O QUE ISSO SIGNIFICA EM PONTOS PARA AMANHÃ:
🟢 Para ser ALTA: Precisa fechar ACIMA de 175486 pts
🔴 Para ser BAIXA: Precisa fechar ABAIXO de 174785 pts
🟡 Zona NEUTRA: Ficar preso entre 174785 e 175486 pts
